In [ ]:
# IMPLEMENTASI JARAK NUMERIK DENGAN NUMPY
import numpy as np
from scipy.spatial.distance import euclidean, cityblock, minkowski
# Data contoh
A	= np.array([2, 3])
B	= np.array([5, 7])

# Perhitungan berbagai jarak
euc_dist = euclidean(A, B)
man_dist = cityblock(A, B)
min_dist = minkowski(A, B, p=3) # r=3
print(f"Euclidean Distance: {euc_dist:.2f}")	# 5.00
print(f"Manhattan Distance: {man_dist:.2f}")	# 7.00
print(f"Minkowski (r=3): {min_dist:.2f}")     #4.50

# Perbandingan dengan data berskala berbeda
A_scaled = np.array([2, 2000000])
B_scaled = np.array([5, 7000000])	# 4.50

print("\nTanpa normalisasi: ")
print(f"Euclidean: {euclidean(A_scaled, B_scaled):.2f}")
#output akan didominasi oleh fitur pendapatan!

Euclidean Distance: 5.00
Manhattan Distance: 7.00
Minkowski (r=3): 4.50

Tanpa normalisasi: 
Euclidean: 5000000.00


In [ ]:
# PENTINGNYA NORMALISASI
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
# Data dengan skala berbeda
data = np.array([[2, 2000000], [5, 7000000], [3,5000000], [8, 12000000]])

# Min-Max Normalization
scaler_minmax = MinMaxScaler()
data_minmax = scaler_minmax.fit_transform(data)

# Z-Score Normalization
scaler_zscore = StandardScaler()
data_zscore = scaler_zscore.fit_transform(data)
print("Data Asli:")
print(data)
print("\nSetelah Min-Max:")
print(data_minmax)
print("\nSetelah Z-Score:")
print(data_zscore)
# Hitung jarak Euclidean setelah normalisasi
from scipy.spatial.distance import euclidean
print(f"\nJarak Euclidean (data asli): {euclidean(data[0], data[1]):.2f}")
print(f"Jarak Euclidean (Min-Max): {euclidean( data_minmax[0], data_minmax[1]):.2f}")



Data Asli:
[[       2  2000000]
 [       5  7000000]
 [       3  5000000]
 [       8 12000000]]

Setelah Min-Max:
[[0.         0.        ]
 [0.5        0.5       ]
 [0.16666667 0.3       ]
 [1.         1.        ]]

Setelah Z-Score:
[[-1.09108945 -1.23624508]
 [ 0.21821789  0.13736056]
 [-0.65465367 -0.41208169]
 [ 1.52752523  1.5109662 ]]

Jarak Euclidean (data asli): 5000000.00
Jarak Euclidean (Min-Max): 0.71


In [ ]:
# SIMILARITAS DATA KATEGORIKAL
import numpy as np
from sklearn.metrics import jaccard_score

def simple_matching_coefficient(p, q):
  """Menghitung SMC untuk dua vektor biner"""
  p = np.array(p)
  q = np.array(q)

  m11 = np.sum((p == 1) & (q == 1))
  m00 = np.sum((p == 0) & (q == 0))
  total = len(p)

  return (m11 + m00) / total

# Data contoh
p = [1, 0, 1, 1, 0]
q = [1, 0, 0, 1, 1]
# Hitung SMC
smc = simple_matching_coefficient(p, q)
print(f"SMC: {smc:.2f}") # 0.60
# Hitung Jaccard (hanya kehadiran)
# M11 = 2, M10 = 1, M01 = 1
m11 = np.sum((np.array(p) == 1) & (np.array(q) == 1))
m10 = np.sum((np.array(p) == 1) & (np.array(q) == 0))
m01 = np.sum((np.array(p) == 0) & (np.array(q) == 1))

jaccard = m11 / (m11 + m10 + m01)
print(f"Jaccard (manual): {jaccard:.2f}") # 0.50

# Alternatif dengan scikit-learn (average=’binary’)
jaccard_sklearn = jaccard_score(p, q, average='binary')
print(f"Jaccard (sklearn): {jaccard_sklearn:.2f}") #0.50


SMC: 0.60
Jaccard (manual): 0.50
Jaccard (sklearn): 0.50


In [1]:
# COSINE SIMILARITY UNTUK TEKS

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Kumpulan dokumen
documents = [
    "data mining machine learning",
    "machine learning data mining",
    "artificial intelligence",
    "deep learning neural network"
]

# Konversi ke vektor TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

# Hitung cosine similarity
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Cosine Similarity Matrix:")
print(similarity_matrix.round(3))

print("\nFitur (kata-kata):")
print(vectorizer.get_feature_names_out())

# Cari dokumen paling mirip dengan dokumen pertama
most_similar = similarity_matrix[0].argsort()[-2]
print(f"\nDokumen 0 paling mirip dengan dokumen {most_similar}")

Cosine Similarity Matrix:
[[1.    1.    0.    0.146]
 [1.    1.    0.    0.146]
 [0.    0.    1.    0.   ]
 [0.146 0.146 0.    1.   ]]

Fitur (kata-kata):
['artificial' 'data' 'deep' 'intelligence' 'learning' 'machine' 'mining'
 'network' 'neural']

Dokumen 0 paling mirip dengan dokumen 1


In [2]:
# LATIHAN: KLASIFIKASI SEDERHANA DENGAN K-NN

import numpy as np
from collections import Counter

def knn_predict(X_train, y_train, X_test, k=3):
    """Prediksi dengan k-NN menggunakan Euclidean distance"""
    predictions = []

    for test_point in X_test:
        # Hitung jarak ke semua data training
        distances = []

        for i, train_point in enumerate(X_train):
            dist = np.sqrt(np.sum((test_point - train_point) ** 2))
            distances.append((dist, y_train[i]))

        # Urutkan berdasarkan jarak terkecil
        distances.sort(key=lambda x: x[0])

        # Ambil k tetangga terdekat
        k_neighbors = distances[:k]
        k_labels = [label for _, label in k_neighbors]

        # Majority voting
        most_common = Counter(k_labels).most_common(1)[0][0]
        predictions.append(most_common)

    return np.array(predictions)

# Contoh data (iris sederhana)
X_train = np.array([
    [5.1, 3.5],
    [4.9, 3.0],
    [6.2, 3.4],
    [5.9, 3.0]
])

y_train = np.array([0, 0, 1, 1])  # 0=setosa, 1=versicolor

X_test = np.array([
    [5.0, 3.2],
    [6.0, 3.2]
])

predictions = knn_predict(X_train, y_train, X_test, k=3)
print(f"Hasil prediksi: {predictions}")  # [0 1]

Hasil prediksi: [0 1]
